<a href="https://colab.research.google.com/github/rjshrd/myrepos/blob/master/multiagentic_code_review.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -qU crewai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.5/42.5 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 3.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.9/211.9 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.8/131.8 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 606.2/606.2 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.4/71.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 88.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/59.5 kB 3.4 MB/s eta 0:00

In [2]:
from crewai import Agent, Task, Crew, LLM
from textwrap import dedent

In [3]:
import os
os.environ["GROQ_API_KEY"] = ""  # Replace with your Groq API key

In [4]:
llm = LLM(model="groq/llama-3.1-70b-versatile")

In [5]:
def get_user_requirements():
    print("\n=== Project Requirements ===")
    project_description = input("Enter the project description: ")
    language = input("Enter the programming language (default is Python): ") or "Python"
    specific_requirements = input("Enter any specific requirements (optional): ")

    # Combine all requirements into a detailed task description
    task_description = f"""
    Create a program with the following specifications:

    Project Description: {project_description}
    Programming Language: {language}
    Additional Requirements: {specific_requirements}

    Please provide:
    1. Well-structured code
    2. Proper documentation
    3. Type hints (if applicable)
    4. Error handling
    5. Examples of usage
    """
    return dedent(task_description)

In [6]:
# Define the Developer Agent
developer = Agent(
    role='Senior Software Developer',
    goal='Write clean, efficient, and well-documented code',
    backstory="""You are an experienced developer with strong skills in
    writing maintainable and efficient code. You take feedback well and are
    committed to continuous improvement.""",
    verbose=True,
    llm=llm
)


In [7]:
# Define the Code Reviewer Agent
reviewer = Agent(
    role='Code Reviewer',
    goal='Ensure code quality and suggest improvements',
    backstory="""You are a meticulous code reviewer with years of experience in
    identifying potential issues and suggesting improvements. You focus on code
    quality, performance, and best practices.""",
    verbose=True,
    llm=llm
)


In [8]:
# Get user requirements
user_task_description = get_user_requirements()


=== Project Requirements ===
Enter the project description: find the largest prime number less than 50
Enter the programming language (default is Python): python
Enter any specific requirements (optional): none


In [9]:
write_code_task = Task(
    description=user_task_description,
    agent=developer,
    expected_output="The expected output is the code generated by the developer agent based on the user's requirements. Don't include any backtick or meta data, just give me the code"
)

In [10]:
# Task 2: Review the code
review_code_task = Task(
    description="""Review the code provided by the developer. Check for:
    1. Code correctness
    2. Performance optimization opportunities
    3. Documentation quality
    4. Type hint usage
    5. Edge cases handling
    Provide specific recommendations for improvement.""",
    agent=reviewer,
    expected_output="Code review feedback and suggestions for improvement.Don't include any backtick or meta data, just give me the code"
)

In [11]:
# Task 3: Implement review feedback
implement_feedback_task = Task(
    description="""Review the feedback provided and implement the suggested
    improvements. Ensure all points are addressed and document any changes
    made.""",
    agent=developer,
    expected_output="Improved code incorporating the review feedback.Don't include any backtick or meta data, just give me the code"
)

In [12]:
# Create the crew
code_crew = Crew(
    agents=[developer, reviewer],
    tasks=[write_code_task, review_code_task, implement_feedback_task],
    verbose=True
)

In [13]:
# Execute the crew's tasks
print("\nStarting the development and review process...")
result = code_crew.kickoff()


Starting the development and review process...
# Agent: Senior Software Developer
## Task: 
Create a program with the following specifications:

Project Description: find the largest prime number less than 50
Programming Language: python
Additional Requirements: none

Please provide:
1. Well-structured code
2. Proper documentation
3. Type hints (if applicable)
4. Error handling
5. Examples of usage



# Agent: Senior Software Developer
## Final Answer: 
# Define a function to check if a number is prime
def is_prime(n: int) -> bool:
    """
    Checks if a number is prime.

    Args:
        n (int): The number to check.

    Returns:
        bool: True if the number is prime, False otherwise.
    """
    if n <= 1:
        return False
    if n == 2:
        return True
    if n % 2 == 0:
        return False
    max_divisor = int(n**0.5) + 1
    for d in range(3, max_divisor, 2):
        if n % d == 0:
            return False
    return True

# Define a function to find the largest 

In [14]:
print(result.raw)

def is_prime(n: int) -> bool:
    """
    Checks if a number is prime.

    Args:
        n (int): The number to check.

    Returns:
        bool: True if the number is prime, False otherwise.

    Raises:
        TypeError: If the input is not an integer.
    """
    if not isinstance(n, int):
        raise TypeError("Input must be an integer")
    if n <= 1:
        return False
    if n == 2:
        return True
    if n % 2 == 0:
        return False
    max_divisor = int(n**0.5) + 1
    for d in range(3, max_divisor, 2):
        if n % d == 0:
            return False
    return True

def largest_prime_less_than(number: int) -> int:
    """
    Finds the largest prime number less than a given number.

    Args:
        number (int): The upper limit.

    Returns:
        int: The largest prime number less than the given number.

    Raises:
        ValueError: If no prime numbers are found less than the given number.
    """
    if number <= 2:
        raise ValueError("No prime 